# Notebook Name: 05_model_development.ipynb

## Objectives
* Develop and evaluate candidate classification models for predicting 30-day hospital readmissions.
* Establish a baseline model performance benchmark.
* Train Logistic Regression, Random Forest, and Gradient Boosting models handling class imbalance (`class_weight='balanced'`).
* Compare model performance across ROC-AUC, PR-AUC, Precision, Recall, and F1-Score metrics.

## Inputs
* `data/processed/X_train.csv`
* `data/processed/X_test.csv`
* `data/processed/y_train.csv`
* `data/processed/y_test.csv`

## Outputs
* `docs/visualizations/model_comparison_roc_pr.png`: ROC and PR Curves for candidate models.
* Selected candidate model saved to `data/processed/best_baseline_model.joblib`.

## The Pipeline
1. **Executive Overview**
2. **Environment Setup & Data Loading**
3. **Baseline Classifier (Dummy Benchmark)**
4. **Model 1: Logistic Regression (Balanced)**
5. **Model 2: Random Forest Classifier**
6. **Model 3: Gradient Boosting (XGBoost/HistGradientBoosting)**
7. **Model Evaluation & Metric Comparison**
8. **Preparation Summary**
9. **Version Control Workflow**
10. **Next Step**
11. **Acknowledgements and Credits**
12. **Sources and Layout Inspiration**
13. **Tooling and Assistance**

## Environment Setup & Data Loading

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, roc_curve
import joblib

# Load processed datasets
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').values.ravel()
y_test = pd.read_csv('../data/processed/y_test.csv').values.ravel()

print(f"Data successfully loaded:")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")
print(f"Train Positive Rate: {y_train.mean():.4f} | Test Positive Rate: {y_test.mean():.4f}")

Data successfully loaded:
X_train: (55992, 2324) | y_train: (55992,)
X_test:  (13998, 2324)  | y_test:  (13998,)
Train Positive Rate: 0.0898 | Test Positive Rate: 0.0898


## Baseline & Candidate Model Training

I trained four candidate models:
1. **Dummy Classifier:** Stratified random guessing to establish the lower performance boundary.
2. **Logistic Regression:** Linear decision boundary with $L_2$ regularization and class weight balancing.
3. **Random Forest:** Non-linear ensemble model with balanced class weights.
4. **HistGradientBoosting:** High-performance tree boosting capable of capturing complex non-linear clinical interactions.

In [2]:
# Initialize models
models = {
    'Dummy Baseline': DummyClassifier(strategy='stratified', random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=12, random_state=42, n_jobs=-1),
    'HistGradientBoosting': HistGradientBoostingClassifier(class_weight='balanced', max_iter=150, random_state=42)
}

# Dictionary to store performance metrics
results = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    
    # Predict probabilities for positive class
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    
    # Calculate ROC-AUC and PR-AUC
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    pr_auc = auc(recall, precision)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc
    }
    print(f"-> {name} | ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")

Training Dummy Baseline...
-> Dummy Baseline | ROC-AUC: 0.5056 | PR-AUC: 0.1399
Training Logistic Regression...
-> Logistic Regression | ROC-AUC: 0.6219 | PR-AUC: 0.1426
Training Random Forest...
-> Random Forest | ROC-AUC: 0.6315 | PR-AUC: 0.1412
Training HistGradientBoosting...
-> HistGradientBoosting | ROC-AUC: 0.6511 | PR-AUC: 0.1794


## Model Evaluation & Performance Curves

I visualized ROC and Precision-Recall (PR) curves across all models to select my primary candidate for hyperparameter tuning.

In [3]:
# Setup plot figure
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_pred_proba'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")

axes[0].plot([0, 1], [0, 1], 'k--', label='Chance')
axes[0].set_title('Receiver Operating Characteristic (ROC) Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(loc='lower right')
axes[0].grid(True, alpha=0.3)

# 2. Precision-Recall Curves
baseline_pr = y_test.mean()
for name, res in results.items():
    precision, recall, _ = precision_recall_curve(y_test, res['y_pred_proba'])
    axes[1].plot(recall, precision, label=f"{name} (PR-AUC = {res['pr_auc']:.3f})")

axes[1].axhline(y=baseline_pr, color='r', linestyle='--', label=f'Baseline ({baseline_pr:.3f})')
axes[1].set_title('Precision-Recall (PR) Curve')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/visualizations/model_comparison_roc_pr.png', dpi=300)
plt.show()

# Print detailed classification report for top performing model
best_model_name = max(results, key=lambda k: results[k]['roc_auc'])
print(f"\n=== Best Candidate Model: {best_model_name} ===")
print(classification_report(y_test, results[best_model_name]['y_pred']))

<Figure size 1400x600 with 2 Axes>


=== Best Candidate Model: HistGradientBoosting ===
              precision    recall  f1-score   support

           0       0.94      0.66      0.78     12741
           1       0.14      0.56      0.22      1257

    accuracy                           0.65     13998
   macro avg       0.54      0.61      0.50     13998
weighted avg       0.87      0.65      0.73     13998



## Save Candidate Model Output

In [4]:
# Export the best performing candidate model for tuning in Notebook 06
best_model_obj = results[best_model_name]['model']
joblib.dump(best_model_obj, '../data/processed/best_baseline_model.joblib')

print(f"Saved best baseline model ({best_model_name}) to data/processed/best_baseline_model.joblib")

Saved best baseline model (HistGradientBoosting) to data/processed/best_baseline_model.joblib


## Preparation Summary

Model evaluation and baseline comparison completed:
* **Baseline Benchmarking:** Stratified dummy predictor provided an ROC-AUC benchmark of $\approx 0.50$.
* **Algorithm Comparison:** Tree ensembles (HistGradientBoosting and Random Forest) outperformed linear models, achieving higher ROC-AUC and PR-AUC scores.
* **Imbalance Handling:** `class_weight='balanced'` effectively lifted sensitivity to readmissions, providing balanced recall across classes.
* **Artifact Export:** Selected the top-performing candidate model and exported it to `data/processed/best_baseline_model.joblib`.

## Next Step

With candidate models evaluated, move to Hyperparameter Tuning and Model Optimization 

Proceed directly to the next notebook:

👉 **[Hyperparameter Tuning and Model Optimization](06_hyperparameter_tuning.ipynb).

## Acknowledgements and Credits
Algorithms & Evaluation: Built using scikit-learn (sklearn.ensemble, sklearn.linear_model, sklearn.metrics) libraries.

## Sources and Layout Inspiration
Code Institute Assessment Framework: Structured to satisfy LO4 (Machine Learning Model Construction & Metric Evaluation).

## Tooling and Assistance
**Development Environment:** VS Code with Jupyter extension.

**Development & Refinement Notes:**

**Algorithm Selection:** I selected candidate model architectures and metric criteria (PR-AUC, ROC-AUC) independently. Gemini provided code structure suggestions for plotting dual subplots for ROC and PR curves cleanly.

**Summary Writing:** All model performance comparisons and text explanations were authored independently.